# GRACE universal potentials in LAMMPS, via `lammpsparser`

This notebook demonstrates how to run the [GRACE](https://gracemaker.readthedocs.io/en/latest/gracemaker/tutorials/#132-lammps) machine-learning interatomic potentials in [LAMMPS](https://www.lammps.org), using the `pair_style grace` implementation contributed by Yury Lysogorskiy and shipped in the `conda-forge` LAMMPS package (`2025.07.22`, build `5`).

Rather than hand-writing a LAMMPS input script, we drive LAMMPS through [`lammpsparser`](https://lammpsparser.readthedocs.io/en/latest/example.html) - a thin, file-based Python wrapper around LAMMPS that many computational-materials users already know from `pyiron`. It writes the LAMMPS data file and input script, runs `lmp_mpi`, and parses the log/dump files back into `numpy` arrays in ASE units.

We use it here in two steps:
1. A single-point (static) energy/force evaluation.
2. A short NVT molecular-dynamics trajectory, with the resulting temperature and energy parsed    and plotted directly from LAMMPS' output files.

## 1. Download a GRACE foundation model

GRACE foundation models are distributed as TensorFlow `SavedModel` directories and fetched with the `grace_models` CLI that ships with the `tensorpotential` package (the training/inference code behind GRACE). We use `GRACE-1L-OAM`, a small single-layer model fitted on OMat24 and fine-tuned on sAlex+MPtrj - light enough to run on CPU in a Binder session.

In [ ]:
!grace_models download GRACE-1L-OAM

## 2. Build the structure and register the potential

`lammpsparser` looks up classical potentials by name in the NIST/OpenKIM database via `get_potential_by_name`, which does not know about GRACE. Instead we hand it a `pandas.Series` with the same shape that function would normally return: a `pair_style`/`pair_coeff` pair pointing at the downloaded `SavedModel` directory.

In [ ]:
import os

import pandas as pd
from ase.build import bulk

model_name = "GRACE-1L-OAM"
model_path = os.path.expanduser(os.path.join("~/.cache/grace", model_name))
element = "Al"

# A small fcc aluminium supercell (32 atoms)
structure = bulk(element, cubic=True).repeat((2, 2, 2))

# lammpsparser normally resolves `potential` via `get_potential_by_name()` against the
# NIST/OpenKIM potential database. GRACE models are not in that database, so we build the
# equivalent `pandas.Series` by hand and pass it in directly.
potential = pd.Series(
    {
        "Name": model_name,
        "Species": [element],
        "Config": [
            "pair_style grace",
            f"pair_coeff * * {model_path} {element}",
        ],
    }
)
potential

## 3. Static energy and force evaluation

`lammps_file_interface_function()` writes the LAMMPS data file and input script, runs `lmp_mpi`, and parses `log.lammps`/`dump.out` back into ASE units.

In [ ]:
from lammpsparser import lammps_file_interface_function

shell_output, static_output, job_crashed = lammps_file_interface_function(
    working_directory="lmp_static",
    structure=structure,
    potential=potential,
    calc_mode="static",
)

assert not job_crashed
print("Potential energy (eV):", static_output["generic"]["energy_pot"][-1])
print("Max. force component (eV/A):", abs(static_output["generic"]["forces"][-1]).max())

## 4. Short NVT molecular-dynamics run

Same potential, `calc_mode="md"`: `lammpsparser` adds the thermostat, velocity initialisation and dump/thermo commands for us.

In [ ]:
shell_output, md_output, job_crashed = lammps_file_interface_function(
    working_directory="lmp_md",
    structure=structure,
    potential=potential,
    calc_mode="md",
    calc_kwargs={
        "temperature": 500.0,
        "n_ionic_steps": 500,
        "n_print": 25,
        "time_step": 1.0,
        "seed": 12345,
    },
)

assert not job_crashed

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

steps = md_output["generic"]["steps"]
temperature = md_output["generic"]["temperature"]
energy_tot = md_output["generic"]["energy_tot"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(steps, temperature, marker="o")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Temperature (K)")
axes[0].set_title("NVT thermostat (target: 500 K)")

axes[1].plot(steps, energy_tot, marker="o", color="tab:orange")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Total energy (eV)")
axes[1].set_title("Energy conservation")

fig.tight_layout()
plt.show()